# 🧰 quant-kit — Universal Quantization Notebook

**Works on: Google Colab ✅ | Kaggle ✅ | Antigravity IDE ✅ | RunPod ✅**

> ⚠️ **TPU Note**: Not supported. llama.cpp / sd.cpp / whisper.cpp require a standard NVIDIA GPU.

### Before you start:
1. `Runtime → Change runtime type → GPU (T4)` (Colab) or `Settings → Accelerator → GPU` (Kaggle)
2. Add your HF token to **Secrets** (🔑 key icon in left sidebar):
   - Colab: Name = `HF_TOKEN` | Value = your token from https://huggingface.co/settings/tokens
   - Kaggle: Add via `Add-ons → Secrets` with the same name

Run cells **top to bottom**.

In [ ]:
# ═══════════════════════════════════════════════════
# Cell 1 — Environment Setup
# Clone repo (skipped if already cloned locally)
# ═══════════════════════════════════════════════════
import os

if not os.path.exists('quant.py'):
    !git clone https://github.com/DhruvalPtl/quant-kit.git
    %cd quant-kit

!pip install -e . -q

# Download binaries for all model types (safe to re-run)
!python setup_linux.py
!python setup_sd_cpp.py
!python setup_whisper_cpp.py

In [ ]:
# ═══════════════════════════════════════════════════
# Cell 2 — HuggingFace Authentication
# Reads token from Colab/Kaggle Secrets — never hardcoded!
# ═══════════════════════════════════════════════════
import os
from huggingface_hub import login

# Auto-detect environment and load token from Secrets
token = None

if os.environ.get('COLAB_GPU') or os.path.exists('/content'):
    # Google Colab — use Colab Secrets
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        print('[OK] Loaded token from Colab Secrets')
    except Exception as e:
        print(f'[!!] Could not load Colab secret: {e}')
        print('     → Click the 🔑 icon → Add secret: Name=HF_TOKEN')

elif os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    # Kaggle — use Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
        print('[OK] Loaded token from Kaggle Secrets')
    except Exception as e:
        print(f'[!!] Could not load Kaggle secret: {e}')
        print('     → Add-ons → Secrets → Add HF_TOKEN')

else:
    # Local / RunPod — read from .env file
    token = os.environ.get('HF_TOKEN')
    if token:
        print(f'[OK] Loaded token from environment')
    else:
        print('[!!] HF_TOKEN not found — make sure your .env file is configured')

if token:
    os.environ['HF_TOKEN'] = token
    login(token=token)
    with open('.env', 'w') as f:
        f.write(f'hf_token = "{token}"\n')
    print('\n✅ Authenticated successfully!')
else:
    print('\n❌ No token found. Set HF_TOKEN in Secrets before continuing.')

## 🚀 Manual Quantization (`quant.py`)
Automatically detects model type and routes to the right backend.

In [ ]:
# ═══════════════════════════════════════════════════
# Cell 3 — Quantize a single model
# Edit MODEL_ID and PRESET before running!
# ═══════════════════════════════════════════════════
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'   # <- change this
PRESET   = 'full'                           # <- standard | full | imatrix

# --delete-src frees disk space after FP16 conversion (critical on Colab/Kaggle)
!python quant.py --model {MODEL_ID} --preset {PRESET} --delete-src

## 🤖 Autopilot (`autopilot.py`)
Fully automated: discover → quantize → model card → upload to HuggingFace.

In [ ]:
# ═══════════════════════════════════════════════════
# Cell 4 — Autopilot (fully automated pipeline)
# ═══════════════════════════════════════════════════

# Option A: Target a specific model
!python autopilot.py --model google/gemma-3-4b-it --max-gb 30

# Option B: Let autopilot discover trending models automatically
# !python autopilot.py --models 3 --max-gb 30